# Financial Sentiment QLoRA Fine-Tuning

This notebook runs the tested training pipeline in `finetune_sentiment.py`. The default base model is `Qwen/Qwen3-4B-Instruct-2507`, replacing the gated Llama 2 7B model and old Git-pinned training libraries.

Use a CUDA GPU runtime. A T4-class GPU should work with the default 4-bit configuration, although batch size may need to be reduced if memory is constrained.

In [ ]:
%pip install -q -r requirements-finetune.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training")

print(torch.cuda.get_device_name(0))
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

## Configure the run

The Financial PhraseBank `sentences_allagree` split uses examples with unanimous annotator agreement. Change `dataset_config` to `sentences_75agree` or `sentences_50agree` to trade label quality for more data.

In [ ]:
from finetune_sentiment import FineTuneConfig

config = FineTuneConfig(
    model_name="Qwen/Qwen3-4B-Instruct-2507",
    dataset_config="sentences_allagree",
    output_dir="artifacts/qwen3-finance-sentiment",
    epochs=3,
    batch_size=4,
    gradient_accumulation_steps=4,
    max_length=256,
).validated()
config

## Train and evaluate

The pipeline creates deterministic stratified train, validation, and test sets. It selects the checkpoint with the lowest validation loss and evaluates the untouched test set with greedy exact-label generation.

In [ ]:
import json
from pathlib import Path

from finetune_sentiment import evaluate_model, train

model, tokenizer, test_dataset = train(config)
metrics = evaluate_model(model, tokenizer, test_dataset)

metrics_path = Path(config.output_dir) / "test_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")
metrics

## Optional: publish the adapter

This publishes the small LoRA adapter rather than a merged full model. Set `HF_REPO_ID` to a repository you own and authenticate with `huggingface_hub.login()` first.

In [ ]:
# from huggingface_hub import login
# login()
# HF_REPO_ID = "your-name/qwen3-finance-sentiment-lora"
# model.push_to_hub(HF_REPO_ID)
# tokenizer.push_to_hub(HF_REPO_ID)